In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
working_directory = "/Users/kemalinecik/git_nosync/sctram"

In [3]:
import sys
sys.path.append(working_directory)

import logging
import os
import numpy as np
import pandas as pd
import networkx as nx
import scanpy as sc
import anndata as ad
from sctram.generate.real import sc_norman_sciplex_cpa

In [4]:
from sctram.api._lower_level import TrajectoryEvaluationAPI
from sctram.input import read_dict

2025-02-07 16:29:38.037 | INFO     | sctram.api._defaults_read:load_default_metrics:23 - Loaded default metrics from /Users/kemalinecik/git_nosync/sctram/sctram/api/_defaults.yaml
2025-02-07 16:29:38.037 | INFO     | sctram.api._defaults_read:load_default_metrics:79 - Default metrics YAML structure validated successfully.


In [5]:
dataset_dir = "/Users/kemalinecik/git_nosync/sctram/__temp__/data"
adata_norman = sc_norman_sciplex_cpa(dataset_dir=dataset_dir)
adata_bms = adata_norman[["bms" in i.lower() or "vehicle" in i.lower() for i in adata_norman.obs["drug"]]]
adata = ad.AnnData(X=adata_bms.obsm["tardis"].copy(), obs=adata_bms.obs.copy())

2025-02-07 16:29:38.064 | WARNING  | sctram.generate.real._download:download_dataset:105 - File PosixPath('/Users/kemalinecik/git_nosync/sctram/__temp__/data/norman_sciplex_cpa.h5ad') already exists. Skipping download.


In [6]:
ground_truth_trajectories = {
    "trajectory_1": [
        ('Vehicle_1.0', 'BMS_0.001'),
        ('BMS_0.001', 'BMS_0.005'),
        ('BMS_0.005', 'BMS_0.01'),
        ('BMS_0.01', 'BMS_0.05'),
        ('BMS_0.05', 'BMS_0.1'),
        ('BMS_0.1', 'BMS_0.5'),
        ('BMS_0.5', 'BMS_1.0'),
    ],
}
input_trajectories_all = read_dict(ground_truth_trajectories)
input_trajectories = input_trajectories_all.get_trajectory("trajectory_1", include_additional_nodes=False)

In [7]:
api = TrajectoryEvaluationAPI(
    adata=adata,
    input_trajectories=input_trajectories,
    labels_obs="drug_dose_name",
    root_label="Vehicle_1.0",
    logger_level="DEBUG"
)

In [8]:
api.evaluate_with_defaults()

2025-02-07 16:29:38.276 | INFO     | sctram.api._lower_level:evaluate_adjacency:150 - Starting adjacency evaluation.
2025-02-07 16:29:38.276 | INFO     | sctram.api._lower_level:_get_inference_method:79 - Running pseudotime inference with method 'PAGAInference'
2025-02-07 16:29:38.276 | INFO     | sctram.api._lower_level:_get_evaluate_method:72 - Running pseudotime evaluation with method 'AdjacencyMatrixEvaluation'
2025-02-07 16:29:38.277 | DEBUG    | sctram.infer._base:_initialize_from_adata_without_neighbors:161 - Initializing from AnnData without precomputed neighbors.
2025-02-07 16:29:38.278 | DEBUG    | sctram.infer._base:_add_labels_to_adata:220 - Adding provided labels to AnnData object.
2025-02-07 16:29:38.279 | INFO     | sctram.infer._base:_initialize_from_adata_without_neighbors:168 - No precomputed neighbors found in AnnData.
2025-02-07 16:29:38.279 | DEBUG    | sctram.infer._base:_initialize_from_adata_without_neighbors:172 - AnnData initialized successfully from AnnData w

Diffusion pseudotime converged in 21 steps.


2025-02-07 16:29:53.547 | DEBUG    | sctram.evaluate._metricsmixin._pseudotimevaluesmetricsmixin:_calculate_dynamic_time_warping:376 - Fallback DTW distance: 0.11527846870098829
2025-02-07 16:29:53.549 | DEBUG    | sctram.evaluate._metricsmixin._pseudotimevaluesmetricsmixin:_calculate:52 - Calculating metric: 'wasserstein_distance'
2025-02-07 16:29:53.551 | DEBUG    | sctram.evaluate._metricsmixin._pseudotimevaluesmetricsmixin:_calculate_wasserstein_distance:401 - Wasserstein distance: 0.1129031260820259
2025-02-07 16:29:53.551 | DEBUG    | sctram.evaluate._metricsmixin._pseudotimevaluesmetricsmixin:_calculate:52 - Calculating metric: 'mutual_information'
2025-02-07 16:29:53.554 | DEBUG    | sctram.evaluate._metricsmixin._pseudotimevaluesmetricsmixin:_calculate:52 - Calculating metric: 'mutual_information_kde'
2025-02-07 16:29:54.179 | DEBUG    | sctram.evaluate._metricsmixin._pseudotimevaluesmetricsmixin:_calculate:52 - Calculating metric: 'cumulative_density_difference'
2025-02-07 16

In [9]:
adata_scvi = ad.AnnData(X=adata_bms.obsm["scvi"].copy(), obs=adata_bms.obs.copy())
api_scvi = TrajectoryEvaluationAPI(
    adata=adata_scvi,
    input_trajectories=input_trajectories,
    labels_obs="drug_dose_name",
    root_label="Vehicle_1.0",
    logger_level="WARNING"
)
api_scvi.evaluate_with_defaults()

2025-02-07 16:30:01.194 | WARNING  | sctram.evaluate._metricsmixin._adjacencymetricsmixin:_calculate_gnn_embedding_distance:618 - GNN Embedding Cosine Similarity method is not tested in depth.
2025-02-07 16:30:01.197 | WARNING  | sctram.evaluate._metricsmixin._adjacencymetricsmixin:_calculate_persistence_diagram_distance:710 - Persistence Diagram Distance method is not tested in depth.
2025-02-07 16:30:01.436 | WARNING  | sctram.utils._utils:sget:33 - Default value 0.5 used for missing key 'alpha'.
2025-02-07 16:30:01.437 | WARNING  | sctram.utils._utils:sget:33 - Default value 100 used for missing key 'n_steps'.
2025-02-07 16:30:01.437 | WARNING  | sctram.utils._utils:sget:33 - Default value 1e-06 used for missing key 'tol'.


Diffusion pseudotime converged in 21 steps.


2025-02-07 16:30:12.924 | ERROR    | sctram.evaluate._metricsmixin._embeddingtrajectorymetricsmixin:_calculate_branch_silhouette_score:666 - Branch Silhouette Score failed: Number of labels is 1. Valid values are 2 to n_samples - 1 (inclusive)
2025-02-07 16:30:13.427 | WARNING  | sctram.evaluate._metricsmixin._embeddingtrajectorymetricsmixin:_calculate_leaf_node_separation:787 - Leaf separation calculation failed: At least two leaves required for separation analysis


In [10]:
df = api.get_all_results()
df_scvi = api_scvi.get_all_results()
df["score_tardis"] = df["score"]
df["score_scvi"] = df_scvi["score"]
del df["score"]
df

,path,metric,score_tardis,score_scvi
0,adjacency,frobenius,2.635132e+00,3.804940e+00
1,adjacency,L1,1.276400e+01,2.307001e+01
2,adjacency,accuracy,2.187500e-01,2.500000e-01
3,adjacency,graph_edit_distance,2.100000e+01,2.100000e+01
4,adjacency,spectral_distance,1.811560e+00,2.844807e+00
5,adjacency,jaccard,2.500000e-01,2.500000e-01
6,adjacency,hamming,5.000000e+01,4.800000e+01
7,adjacency,precision,7.500000e-01,6.666667e-01
8,adjacency,recall,4.285714e-01,5.714286e-01
9,adjacency,f1_score,5.454545e-01,6.153846e-01
